In [2]:
# single-cell analysis package
library(Seurat)
library(harmony)

# plotting and data science packages
library(repr)
library(stringr)
library(dplyr) 
library(tidyverse)
library(ggplot2)
library(cowplot)
library(patchwork)

# set random seed for reproducibility
set.seed(12345)

In [3]:
# Output Directory
outDir <- file.path("./results")

# load the unprocessed dataset
stObj <- readRDS(file.path('./../data/unprocessed_STdata.rds'))
print(stObj)
print(table(stObj@meta.data$patient))
print(table(stObj@meta.data$chemotherapy_response))

An object of class Seurat 
33538 features across 20123 samples within 1 assay 
Active assay: Spatial (33538 features, 0 variable features)
 8 layers present: counts.1, counts.2, counts.3, counts.4, counts.5, counts.6, counts.7, counts.8
 8 spatial fields of view present: P1 P2 P3 P4 P5 P6 P7 P8

  P1   P2   P3   P4   P5   P6   P7   P8 
2940 1935 1504 3007 2124 1801 3632 3180 

   good partial    poor 
   5563    4808    9752 


In [4]:
print(table(stObj@meta.data$patient, stObj@meta.data$chemotherapy_response))

    
     good partial poor
  P1    0       0 2940
  P2 1935       0    0
  P3 1504       0    0
  P4    0    3007    0
  P5 2124       0    0
  P6    0    1801    0
  P7    0       0 3632
  P8    0       0 3180


## Add image coordinates

In [5]:
data_dir = "./../data/st_data"
dir_list = list.dirs(data_dir, recursive = FALSE)

# Initialize an empty list to store the data frames
df_list <- list()

for (d in dir_list){
    pat <- sub(".*SP", "P", d)
    tissue_pos <- read.csv(paste0(d, "/spatial/tissue_positions_list.csv"), header = FALSE)
    colnames(tissue_pos) <- c("barcode", "tissue", "row", "col", "imagerow", "imagecol")
    tissue_pos <- tissue_pos %>% mutate(pat_barcode = paste0(pat, "_", barcode))
    rownames(tissue_pos) <- tissue_pos$pat_barcode
    print(dim(tissue_pos))
    df_list[[substr(pat, 2, 2)]] <- tissue_pos
    }

# Combine all data frames in the list into a single data frame
tissue_positions_df <- bind_rows(df_list)
tissue_positions_df <- tissue_positions_df[intersect(rownames(tissue_positions_df), rownames(stObj@meta.data)), ]
print(dim(tissue_positions_df))

# Check rownames of metadat is matching with names of vector
print(table(rownames(stObj@meta.data) == rownames(tissue_positions_df)))

[1] 4992    7
[1] 4992    7
[1] 4992    7
[1] 4992    7
[1] 4992    7
[1] 4992    7
[1] 4992    7
[1] 4992    7
[1] 20123     7

FALSE  TRUE 
20120     3 


In [6]:
# join the image_df with the Seurat metadata
stObj@meta.data$pat_barcode <- rownames(stObj@meta.data)
new_meta <- dplyr::left_join(stObj@meta.data, tissue_positions_df, by='pat_barcode')
# Check rownames of metadat is matching with names of vector
print(table(rownames(stObj@meta.data) == new_meta$pat_barcode))


 TRUE 
20123 


In [7]:
## Add image coordinates
stObj@meta.data$row <- new_meta$row
stObj@meta.data$col <- new_meta$col
stObj@meta.data$imagerow <- new_meta$imagerow
stObj@meta.data$imagecol <- new_meta$imagecol

## Quality Control

In [8]:
cellType_annotations <- as.data.frame(read.csv("./../cellType_deconvolution/results/rctd_cell_type_annotations.csv"))
rownames(cellType_annotations) <- cellType_annotations$index
#head(cellType_annotations, 5)

print(length(intersect(cellType_annotations$index, rownames(stObj@meta.data))))
cellType_annotations <- cellType_annotations[rownames(stObj@meta.data),]
print(dim(cellType_annotations))

cellType <- setNames(cellType_annotations$cell_type_annotations, cellType_annotations$index)
stObj$cell_type <- NA  # Initialize with NA

# Map values to the Seurat object metadata
stObj$cell_type[names(cellType) %in% rownames(stObj@meta.data)] <- cellType[names(cellType) %in% rownames(stObj@meta.data)]
print(table(rownames(stObj@meta.data) == names(cellType)))
rows_na_column <- which(is.na(stObj@meta.data$cell_type))
print("Rows with NA values in cell type")
print(rows_na_column)
table(stObj$cell_type)

[1] 20109
[1] 20123     2

 TRUE 
20109 
[1] "Rows with NA values in cell type"
 [1]   341  2492  2521  3172  6468  6747  8925  9634 10586 11202 13031 13750
[13] 15386 16380



   B_Plasma_cells Endothelial_cells       Fibroblasts       Macrophages 
              495               507              3812              2379 
Mesothelial_cells    Myofibroblasts           T_cells      Tumour_cells 
               23              1362                13             11518 

In [9]:
# Remove rows with NA in cell type
stObj <- stObj[, !is.na(stObj$cell_type)]
print(stObj)

# Remove underrepresented cell types
filtered_barcode_index <- rownames(filter(stObj@meta.data, cell_type != "T_cells" & cell_type != "Mesothelial_cells" ))
stObj <- subset(stObj, cells = filtered_barcode_index)
print(stObj)

# QC
stObj <- subset(stObj, subset = nCount_Spatial > 500 )
print(stObj)
print(table(stObj@meta.data$patient))
print(table(stObj@meta.data$chemotherapy_response))

Warning message:
“Not validating Centroids objects”
Warning message:
“Not validating Centroids objects”
Warning message:
“Not validating FOV objects”
Warning message:
“Not validating FOV objects”
Warning message:
“Not validating FOV objects”
Warning message:
“Not validating FOV objects”
Warning message:
“Not validating FOV objects”
Warning message:
“Not validating FOV objects”
Warning message:
“Not validating Seurat objects”
Warning message:
“Not validating Centroids objects”
Warning message:
“Not validating Centroids objects”
Warning message:
“Not validating FOV objects”
Warning message:
“Not validating FOV objects”
Warning message:
“Not validating FOV objects”
Warning message:
“Not validating FOV objects”
Warning message:
“Not validating FOV objects”
Warning message:
“Not validating FOV objects”
Warning message:
“Not validating Seurat objects”
Warning message:
“Not validating Centroids objects”
Warning message:
“Not validating Centroids objects”
Warning message:
“Not validating FOV o

An object of class Seurat 
33538 features across 20109 samples within 1 assay 
Active assay: Spatial (33538 features, 0 variable features)
 8 layers present: counts.1, counts.2, counts.3, counts.4, counts.5, counts.6, counts.7, counts.8
 8 spatial fields of view present: P1 P2 P3 P4 P5 P6 P7 P8


Warning message:
“Not validating Centroids objects”
Warning message:
“Not validating Centroids objects”
Warning message:
“Not validating FOV objects”
Warning message:
“Not validating FOV objects”
Warning message:
“Not validating FOV objects”
Warning message:
“Not validating FOV objects”
Warning message:
“Not validating FOV objects”
Warning message:
“Not validating FOV objects”
Warning message:
“Not validating Seurat objects”
Warning message:
“Not validating Centroids objects”
Warning message:
“Not validating Centroids objects”
Warning message:
“Not validating FOV objects”
Warning message:
“Not validating FOV objects”
Warning message:
“Not validating FOV objects”
Warning message:
“Not validating FOV objects”
Warning message:
“Not validating FOV objects”
Warning message:
“Not validating FOV objects”
Warning message:
“Not validating Seurat objects”
Warning message:
“Not validating Centroids objects”
Warning message:
“Not validating Centroids objects”
Warning message:
“Not validating FOV o

An object of class Seurat 
33538 features across 20073 samples within 1 assay 
Active assay: Spatial (33538 features, 0 variable features)
 8 layers present: counts.1, counts.2, counts.3, counts.4, counts.5, counts.6, counts.7, counts.8
 8 spatial fields of view present: P1 P2 P3 P4 P5 P6 P7 P8


Warning message:
“Not validating Centroids objects”
Warning message:
“Not validating Centroids objects”
Warning message:
“Not validating FOV objects”
Warning message:
“Not validating FOV objects”
Warning message:
“Not validating FOV objects”
Warning message:
“Not validating FOV objects”
Warning message:
“Not validating FOV objects”
Warning message:
“Not validating FOV objects”
Warning message:
“Not validating Seurat objects”
Warning message:
“Not validating Centroids objects”
Warning message:
“Not validating Centroids objects”
Warning message:
“Not validating FOV objects”
Warning message:
“Not validating FOV objects”
Warning message:
“Not validating FOV objects”
Warning message:
“Not validating FOV objects”
Warning message:
“Not validating FOV objects”
Warning message:
“Not validating FOV objects”
Warning message:
“Not validating Seurat objects”
Warning message:
“Not validating Centroids objects”
Warning message:
“Not validating Centroids objects”
Warning message:
“Not validating FOV o

An object of class Seurat 
33538 features across 19996 samples within 1 assay 
Active assay: Spatial (33538 features, 0 variable features)
 8 layers present: counts.1, counts.2, counts.3, counts.4, counts.5, counts.6, counts.7, counts.8
 8 spatial fields of view present: P1 P2 P3 P4 P5 P6 P7 P8

  P1   P2   P3   P4   P5   P6   P7   P8 
2925 1920 1498 3002 2104 1792 3589 3166 

   good partial    poor 
   5522    4794    9680 


In [10]:
# rejoin layers
stObj <- JoinLayers(stObj)
stObj

An object of class Seurat 
33538 features across 19996 samples within 1 assay 
Active assay: Spatial (33538 features, 0 variable features)
 1 layer present: counts
 8 spatial fields of view present: P1 P2 P3 P4 P5 P6 P7 P8

## Normalisation

In [11]:
stObj <- NormalizeData(stObj, assay = "Spatial", verbose = FALSE)
stObj <- FindVariableFeatures(stObj, selection.method = "vst", nfeatures = 3000, verbose = FALSE)
stObj <- ScaleData(stObj, features = VariableFeatures(stObj), verbose = FALSE)
print(stObj)

An object of class Seurat 
33538 features across 19996 samples within 1 assay 
Active assay: Spatial (33538 features, 3000 variable features)
 3 layers present: counts, data, scale.data
 8 spatial fields of view present: P1 P2 P3 P4 P5 P6 P7 P8


In [12]:
stObj <- RunPCA(stObj, npcs = 50, verbose = FALSE)
stObj <- RunHarmony(stObj, group.by.vars='patient',assay='Spatial')
stObj <- RunUMAP(stObj, reduction = "harmony", dims = 1:30, verbose = FALSE)
stObj <- RunTSNE(stObj, reduction = "harmony", dims = 1:30, verbose = FALSE)

Transposing data matrix

Initializing state using k-means centroids initialization

Harmony 1/10

Harmony 2/10

Harmony 3/10

Harmony converged after 3 iterations

Warning message:
“The default method for RunUMAP has changed from calling Python UMAP via reticulate to the R-native UWOT using the cosine metric
To use Python UMAP via reticulate, set umap.method to 'umap-learn' and metric to 'correlation'
This message will be shown once per session”


In [13]:
stObj <- FindNeighbors(stObj, reduction = "harmony", dims = 1:30, verbose = FALSE)
stObj <- FindClusters(stObj, resolution = 0.5, verbose = FALSE)
print(stObj)

An object of class Seurat 
33538 features across 19996 samples within 1 assay 
Active assay: Spatial (33538 features, 3000 variable features)
 3 layers present: counts, data, scale.data
 4 dimensional reductions calculated: pca, harmony, umap, tsne
 8 spatial fields of view present: P1 P2 P3 P4 P5 P6 P7 P8


In [14]:
saveRDS(stObj, paste0( "./results/",'HGSOC_ST_seurat_processed.rds'))